# Tool Schema Design: Fix-the-Schema Practice

## Table of contents
- [Why schema design matters](#why-schema-design-matters)
- [input_schema field types](#input_schema-field-types)
- [Design checklist](#design-checklist)
- [Setup](#setup)
- [Part 1: Five broken schemas](#part-1-five-broken-schemas--find-the-flaw)
- [Part 2: Live demo](#part-2-live-demo)
- [Exam practice](#exam-practice)
- [Quick reference](#quick-reference)

## Why schema design matters

Schema quality directly controls Claude's tool-calling accuracy.
A poorly designed schema leads to wrong calls, missed fields, or hallucinated values.

**Pattern (high frequency):**
> "Given a tool schema, identify the flaw."

The exam typically presents four schema options — one missing `required`, one with vague
descriptions, one with a type error, one correct. This notebook covers the 5 most common
flaws so you can spot them instantly.

## `input_schema` field types

| Value you need | `"type"` | Extra keys | Example |
|---|---|---|---|
| Text | `"string"` | `"enum"` to restrict values | `{ "type": "string", "enum": ["asc", "desc"] }` |
| Whole number | `"integer"` | `"minimum"`, `"maximum"` | `{ "type": "integer" }` |
| Decimal | `"number"` | — | `{ "type": "number" }` (confidence 0.0–1.0) |
| True/false flag | `"boolean"` | — | `{ "type": "boolean" }` |
| Ordered list | `"array"` | **`"items"` required** | `{ "type": "array", "items": { "type": "string" } }` |
| Nested object | `"object"` | `"properties"`, `"required"` | `{ "type": "object", "properties": { ... } }` |

> **Exam trap**: `"integer"` ≠ `"number"`.
> Use `"integer"` for counts and ratings (1–5); `"number"` for decimals (0.0–1.0).

In [ ]:
// Six field types with realistic property definitions
const fieldTypeExamples: Record<string, unknown> = {
  string_with_enum: {
    type: 'string',
    description: 'Sort direction',
    enum: ['asc', 'desc'],
  },
  integer: {
    type: 'integer',
    description: 'Rating from 1 to 5',
    minimum: 1,
    maximum: 5,
  },
  number: {
    type: 'number',
    description: 'Confidence score between 0.0 and 1.0',
  },
  boolean: {
    type: 'boolean',
    description: 'Include archived records in results',
  },
  array: {
    type: 'array',
    description: 'List of user IDs to notify',
    items: { type: 'string' },
  },
  object: {
    type: 'object',
    description: 'Filter criteria for the query',
    properties: {
      status: { type: 'string', description: 'Record status filter' },
    },
    required: ['status'],
  },
};

for (const [typeName, schema] of Object.entries(fieldTypeExamples)) {
  const s = schema as { type: string };
  console.log(`${typeName.padEnd(18)}: type = ${JSON.stringify(s.type)}`);
}

## Design checklist

Before finalizing any tool schema, verify all rules:

| # | Rule | Why it matters |
|---|---|---|
| 1 | Tool `description` says **when** to call it | Claude decides whether to call based on this alone |
| 2 | Every property has a `description` | Claude fills fields using property descriptions |
| 3 | `required` array is present and complete | Fields not listed may be silently omitted |
| 4 | Every `array` field has `items` with a type | Without `items`, Claude doesn't know element types |
| + | Fixed-value strings use `enum` | Prevents Claude from hallucinating invalid values |
| + | Whole-number fields use `"integer"` not `"number"` | Avoids `3.7` when `3` is expected |

## Setup

No extra package needed — this notebook uses only `@anthropic-ai/sdk`.
Ensure `ANTHROPIC_API_KEY` is in `../.env`.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });

console.log('Client ready. Model:', MODEL_NAME);

## Part 1: Five broken schemas — find the flaw

Each schema below has exactly one flaw.
Read it, identify the problem, then scroll to the next cell for the fix.

| # | Flaw | Effect on Claude |
|---|---|---|
| 1 | Missing `required` | May skip mandatory fields |
| 2 | Missing property `description` | Guesses (often wrong) what to put in the field |
| 3 | Vague tool `description` | Doesn't know when — or whether — to call the tool |
| 4 | `array` without `items` | Doesn't know what type array elements should be |
| 5 | `"number"` vs `"integer"`, no `enum` | May send `3.7` for a 1–5 rating or invent invalid values |

In [ ]:
// Looser interface so TypeScript does not block intentionally broken schemas
interface ToolSchema {
  name: string;
  description: string;
  input_schema: Record<string, unknown>;
}

// ── FLAW 1: Missing `required` ───────────────────────────────────────────────
const broken1: ToolSchema = {
  name: 'search_documents',
  description:
    'Search through documents by keyword and date range. ' +
    'Use when the user asks to find documents.',
  input_schema: {
    type: 'object',
    properties: {
      keyword:    { type: 'string', description: 'Search keyword' },
      start_date: { type: 'string', description: 'Start date in YYYY-MM-DD format' },
    },
    // ❌ FLAW: no "required" array — Claude may omit keyword entirely
  },
};

// ── FLAW 2: Missing property descriptions ────────────────────────────────────
const broken2: ToolSchema = {
  name: 'create_ticket',
  description:
    'Create a support ticket. Use when the user wants to open a new support request.',
  input_schema: {
    type: 'object',
    properties: {
      title:    { type: 'string' },  // ❌ FLAW: no description
      priority: { type: 'string' },  // ❌ FLAW: no description, no enum
      category: { type: 'string' },  // ❌ FLAW: no description
    },
    required: ['title', 'priority'],
  },
};

// ── FLAW 3: Vague tool description ───────────────────────────────────────────
const broken3: ToolSchema = {
  name: 'weather',
  description: 'Weather tool',  // ❌ FLAW: Claude cannot determine when to call this
  input_schema: {
    type: 'object',
    properties: {
      city: { type: 'string', description: 'City name, e.g. "Tokyo"' },
    },
    required: ['city'],
  },
};

// ── FLAW 4: array without items ──────────────────────────────────────────────
const broken4: ToolSchema = {
  name: 'send_notifications',
  description:
    'Send a message to a list of users. Use when broadcasting a notification.',
  input_schema: {
    type: 'object',
    properties: {
      user_ids: { type: 'array', description: 'List of user IDs to notify' }, // ❌ FLAW: no items
      message:  { type: 'string', description: 'Notification message body' },
    },
    required: ['user_ids', 'message'],
  },
};

// ── FLAW 5: "number" instead of "integer"; no enum for fixed values ───────────
const broken5: ToolSchema = {
  name: 'rate_response',
  description:
    'Rate an AI response on a scale of 1 to 5 and classify its sentiment. ' +
    'Use after presenting an AI-generated answer.',
  input_schema: {
    type: 'object',
    properties: {
      score:     { type: 'number', description: 'Rating from 1 to 5' },             // ❌ FLAW: should be integer
      sentiment: { type: 'string', description: 'positive, neutral, or negative' }, // ❌ FLAW: should use enum
    },
    required: ['score', 'sentiment'],
  },
};

const broken = [broken1, broken2, broken3, broken4, broken5];
console.log('=== Five broken schemas (spot the flaw!) ===');
broken.forEach((t, i) => console.log(`Broken ${i + 1}: ${t.name}`));

In [ ]:
// Fixed versions — compare each change to the broken schema in the previous cell

// ── FIX 1: Add required ──────────────────────────────────────────────────────
const fixed1: ToolSchema = {
  name: 'search_documents',
  description:
    'Search through documents by keyword and date range. ' +
    'Use when the user asks to find documents.',
  input_schema: {
    type: 'object',
    properties: {
      keyword:    { type: 'string', description: 'Search keyword' },
      start_date: { type: 'string', description: 'Start date in YYYY-MM-DD format' },
    },
    required: ['keyword'],  // ✅ FIX: keyword mandatory; start_date stays optional
  },
};

// ── FIX 2: Add descriptions (and enum) to every property ─────────────────────
const fixed2: ToolSchema = {
  name: 'create_ticket',
  description:
    'Create a support ticket. Use when the user wants to open a new support request.',
  input_schema: {
    type: 'object',
    properties: {
      title:    { type: 'string', description: 'Short summary of the issue' },                                   // ✅ FIX
      priority: { type: 'string', description: 'Priority level', enum: ['low', 'medium', 'high'] },             // ✅ FIX + enum
      category: { type: 'string', description: 'Support category, e.g. "billing" or "technical"' },            // ✅ FIX
    },
    required: ['title', 'priority'],
  },
};

// ── FIX 3: Specific description with trigger phrase ───────────────────────────
const fixed3: ToolSchema = {
  name: 'get_weather',  // ✅ clearer name
  description:
    'Get the current temperature and weather conditions for a city. ' +
    'Use this when the user asks about current weather or temperature.',  // ✅ FIX: trigger phrase
  input_schema: {
    type: 'object',
    properties: {
      city: { type: 'string', description: 'City name, e.g. "Tokyo"' },
    },
    required: ['city'],
  },
};

// ── FIX 4: Add items to array ────────────────────────────────────────────────
const fixed4: ToolSchema = {
  name: 'send_notifications',
  description:
    'Send a message to a list of users. Use when broadcasting a notification.',
  input_schema: {
    type: 'object',
    properties: {
      user_ids: {
        type: 'array',
        description: 'List of user IDs to notify',
        items: { type: 'string' },  // ✅ FIX: element type declared
      },
      message: { type: 'string', description: 'Notification message body' },
    },
    required: ['user_ids', 'message'],
  },
};

// ── FIX 5: integer + enum ────────────────────────────────────────────────────
const fixed5: ToolSchema = {
  name: 'rate_response',
  description:
    'Rate an AI response on a scale of 1 to 5 and classify its sentiment. ' +
    'Use after presenting an AI-generated answer.',
  input_schema: {
    type: 'object',
    properties: {
      score: {
        type: 'integer',      // ✅ FIX: integer, not number
        description: 'Rating from 1 to 5',
        minimum: 1,
        maximum: 5,
      },
      sentiment: {
        type: 'string',
        description: 'Overall sentiment of the AI response',
        enum: ['positive', 'neutral', 'negative'],  // ✅ FIX: enum constrains values
      },
    },
    required: ['score', 'sentiment'],
  },
};

const fixSummaries = [
  'Added required: ["keyword"]',
  'Added description to all properties; added enum to priority',
  'Rewrote description with trigger phrase; renamed to get_weather',
  'Added items: { type: "string" } to user_ids array',
  'Changed score to "integer"; added enum to sentiment',
];

console.log('=== Fixed schemas — changes from broken versions ===');
fixSummaries.forEach((fix, i) => console.log(`Fix ${i + 1}: ${fix}`));

## Part 2: Live demo — Claude with a well-designed schema

The `analyze_sentiment` tool below applies every rule from the checklist:

| Rule | Applied |
|---|---|
| Specific description + trigger phrase | "Use this when the user asks for sentiment analysis..." |
| Every property has a `description` | All 5 properties described |
| `required` is complete | All 5 fields required |
| `array` has `items` | `key_phrases` items typed as `string` |
| `enum` for fixed-value strings | sentiment: positive / neutral / negative |
| `integer` for whole-number rating | rating: 1–5 |
| `number` for decimal confidence | confidence: 0.0–1.0 |

Claude will call this tool to analyze a product review.

In [ ]:
const analyzeTool: Anthropic.Tool = {
  name: 'analyze_sentiment',
  description:
    'Analyze the sentiment, quality, and key themes of a piece of text. ' +
    'Use this when the user asks for sentiment analysis or text evaluation.',
  input_schema: {
    type: 'object',
    properties: {
      sentiment: {
        type: 'string',
        description: 'Overall sentiment of the text',
        enum: ['positive', 'neutral', 'negative'],
      },
      confidence: {
        type: 'number',
        description: 'Confidence score for the sentiment classification, 0.0 to 1.0',
      },
      rating: {
        type: 'integer',
        description: 'Overall quality rating from 1 (lowest) to 5 (highest)',
        minimum: 1,
        maximum: 5,
      },
      key_phrases: {
        type: 'array',
        description: 'Key phrases from the text that influenced the sentiment',
        items: { type: 'string' },
      },
      requires_followup: {
        type: 'boolean',
        description: 'True if the text contains issues requiring human follow-up',
      },
    },
    required: ['sentiment', 'confidence', 'rating', 'key_phrases', 'requires_followup'],
  },
};

function runAnalysis(input: {
  sentiment: string;
  confidence: number;
  rating: number;
  key_phrases: string[];
  requires_followup: boolean;
}) {
  return { ...input, analyzed_at: new Date().toISOString() };
}

const messages: Anthropic.MessageParam[] = [{
  role: 'user',
  content:
    'Please analyze the sentiment of this review: ' +
    '"The product exceeded my expectations! Fast delivery, great quality, ' +
    'and the customer support team resolved my question within an hour."',
}];

while (true) {
  const resp = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 512,
    tools: [analyzeTool],
    messages,
  });

  console.log('stop_reason:', resp.stop_reason);

  if (resp.stop_reason === 'end_turn') {
    const text = resp.content.find(b => b.type === 'text');
    console.log('\nFinal answer:', text?.type === 'text' ? text.text : '(no text)');
    break;
  }

  messages.push({ role: 'assistant', content: resp.content });

  const results: Anthropic.ToolResultBlockParam[] = [];
  for (const block of resp.content) {
    if (block.type !== 'tool_use') continue;
    console.log('tool_use input:\n', JSON.stringify(block.input, null, 2));
    const result = runAnalysis(block.input as Parameters<typeof runAnalysis>[0]);
    console.log('tool_result:\n', JSON.stringify(result, null, 2));
    results.push({ type: 'tool_result', tool_use_id: block.id, content: JSON.stringify(result) });
  }
  messages.push({ role: 'user', content: results });
}

## Exam practice

Five schema scenarios — each has exactly one flaw.
Run the cell to see the flaw and the fix.

In [ ]:
interface ExamScenario {
  schemaName: string;
  flaw: string;
  fix: string;
}

const examScenarios: ExamScenario[] = [
  {
    schemaName: 'delete_record',
    flaw: 'Missing "required" — Claude may skip record_id and the confirm flag',
    fix:  'Add required: ["record_id", "confirm"]',
  },
  {
    schemaName: 'translate_text',
    flaw: 'Vague description "Translate text." — Claude does not know when to call it',
    fix:  'Add "Use this when the user asks to translate text into another language."',
  },
  {
    schemaName: 'bulk_tag',
    flaw: 'item_ids array is missing "items" — Claude does not know the element type',
    fix:  'Add items: { type: "string" } to item_ids',
  },
  {
    schemaName: 'set_priority',
    flaw: 'priority is a plain string — should use enum: ["low","medium","high"]',
    fix:  'Add enum: ["low", "medium", "high"] to the priority property',
  },
  {
    schemaName: 'score_submission',
    flaw: 'Two flaws: score type should be "integer"; feedback property has no description',
    fix:  'Change score to "integer"; add description to feedback',
  },
];

console.log('=== Exam Practice: Find the Flaw ===\n');
examScenarios.forEach((s, i) => {
  console.log(`Scenario ${i + 1}: ${s.schemaName}`);
  console.log(`  Flaw : ${s.flaw}`);
  console.log(`  Fix  : ${s.fix}`);
  console.log();
});

## Quick reference

### Schema design checklist

```
✓ Tool description says WHEN to call (trigger phrase: "Use this when...")
✓ Every property has a description
✓ required lists all mandatory fields
✓ array fields have items with a type
✓ Fixed-value strings use enum
✓ Whole numbers use "integer", decimals use "number"
```

### Five exam-high-frequency flaws

| Flaw | Effect | Fix |
|---|---|---|
| Missing `required` | Claude skips fields | Add `required: [...]` |
| Missing property `description` | Claude guesses wrong values | Add `description` to each property |
| Vague tool `description` | Claude misses trigger or calls at wrong time | Add "Use this when..." trigger phrase |
| `array` without `items` | Claude doesn't know element type | Add `items: { type: "..." }` |
| `"number"` for whole-number field | Claude sends `3.7` instead of `3` | Change to `"integer"` |
| Plain `string` for fixed values | Claude hallucinates invalid values | Add `enum: [...]` |

### Exam rule

> If a scenario asks "what is wrong with this schema":
> 1. Check tool `description` — specific? trigger phrase present?
> 2. Check `required` — present and complete?
> 3. Check every `array` — does it have `items`?
> 4. Check types — `"integer"` vs `"number"`? Fixed values → `"enum"`?
> 5. Check every property — does each one have a `description`?

### Connection to other notebooks

- `00_mcp_vs_tool_use.ipynb` — schema structure is identical for raw tool_use and MCP Tools
- `01_mcp_primitives.ipynb` — MCP Tool's `inputSchema` maps 1:1 to `input_schema` here
- `04_mcp_client_with_claude.ipynb` — schema quality affects error rates in production